In [14]:
"""
TREND/Leachman reference points
"""

import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator, LogLocator, 
    FixedLocator, FuncFormatter, FixedFormatter)
from h2_thermo_final import (R, EOS_PARAMS, eos_ac_b, a_of_T, molar_volume_PR,
    real_properties, sonic_velocity, joule_thomson)

OUTDIR = "TREND Leachman figures"
SPECIES = "p"  # pH2
Mw_H2 = 2.01588e-3  # [kg/mol]


# --- copied from plot_real_state_figures.py --------------------------------------
def fugacity_coeff(T, P, species, phase):
    ac, b, m, Tc, Cpen = eos_ac_b(species)
    a = a_of_T(T, species)
    V_corr, Z = molar_volume_PR(T, P, species, phase)
    A = a * P / (R*T)**2
    B = b * P / (R*T)
    sqrt2 = np.sqrt(2)
    ln_phi = ((Z - 1) - np.log(Z - B)
              - (A/(2*sqrt2*B)) * np.log((Z + (1+sqrt2)*B) / (Z + (1-sqrt2)*B)))
    return np.exp(ln_phi)

def saturation_pressure(T, species, P_guess=None, tol=1e-10, max_iter=300):
    Pc = EOS_PARAMS[species]["Pc"]
    Tc = EOS_PARAMS[species]["Tc"]
    omega = EOS_PARAMS[species]["omega"]
    if P_guess is None:
        Tr = T / Tc
        P_guess = Pc * 10**(7.0/3.0*(1+omega)*(1-1/Tr))
    P = P_guess
    for _ in range(max_iter):
        phi_l = fugacity_coeff(T, P, species, "liquid")
        phi_v = fugacity_coeff(T, P, species, "vapor")
        ratio = phi_l / phi_v
        P_new = P * ratio
        if abs(ratio - 1) < tol:
            return P_new
        P = P_new
    return P

def T_saturation(P_target, species, T_bracket=None):
    Tc_ = EOS_PARAMS[species]["Tc"]
    Tt_ = EOS_PARAMS[species]["Tt"]
    if P_target >= EOS_PARAMS[species]["Pc"]:
        return None
    f = lambda T: saturation_pressure(T, species) - P_target
    lo, hi = Tt_, Tc_ - 0.01
    try:
        return brentq(f, lo, hi, xtol=1e-6)
    except ValueError:
        return None

def build_T_grid(Tsat):
    coarse = np.geomspace(15, 600, 200)
    if Tsat is None or not (15 < Tsat < 600):
        return np.sort(coarse)
    below = Tsat - np.geomspace(0.1, min(5.0, Tsat - 15), 80)
    above = Tsat + np.geomspace(0.1, min(5.0, 600 - Tsat), 80)
    grid = np.concatenate([coarse, below, above])
    grid = grid[(grid > 15) & (grid < 600)]
    return np.unique(np.sort(grid))

def scan_property(P_bar, prop_func):
    P = P_bar * 1e5
    Tsat = T_saturation(P, SPECIES)
    T_out, V_out = [], []
    T_grid = build_T_grid(Tsat)
    for T in T_grid:
        if Tsat is not None and T < Tsat:
            phase = "liquid"
        else:
            phase = "vapor"
        try:
            val = prop_func(T, P, phase)
        except Exception:
            val = np.nan
        T_out.append(T)
        V_out.append(val)
    return np.array(T_out), np.array(V_out), Tsat

def get_density(T, P, phase):
    props = real_properties(T, P, SPECIES, phase=phase)
    return Mw_H2 / props["V_corr"]
def get_Cp_over_R(T, P, phase):
    return real_properties(T, P, SPECIES, phase=phase)["Cp"] / R
def get_Cv_over_R(T, P, phase):
    return real_properties(T, P, SPECIES, phase=phase)["Cv"] / R
def get_sonic(T, P, phase):
    return sonic_velocity(T, P, SPECIES, phase=phase)
def get_JT(T, P, phase):
    return joule_thomson(T, P, SPECIES, phase=phase) * 1e5  # [K/Pa] -> [K/bar]


# --- load TREND/Leachman reference points --------------------------------------
with open("trend_leachman_reference.json") as f:
    trend_data = json.load(f)
	

pressures_bar = [0.1, 1, 10, 20]


def make_4panel_with_overlay(prop_func, trend_key, ylabel, title, fname, y_specs, divide_by_R=False):
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, P_bar, tag, (ymin, ymax, ystep) in zip(axes.flat, pressures_bar,
                                                     ["(a)", "(b)", "(c)", "(d)"], y_specs):
        # our model curve
        T_out, V_out, Tsat = scan_property(P_bar, prop_func)
        ax.plot(T_out, V_out, color="tab:orange", lw=1.5, label="This work (PR-78)", zorder=2)
        if Tsat is not None:
            ax.axvline(Tsat, color="gray", ls=":", lw=1)

        # TREND/Leachman reference points
        row = trend_data[next(k for k in trend_data if float(k) == P_bar)]
        T_ref = np.array(row["T"])
        V_ref = np.array(row[trend_key])
        if divide_by_R:
            V_ref = V_ref / R
        ax.scatter(T_ref, V_ref, s=14, color="tab:blue", zorder=3, label="TREND (Leachman EOS)")

        ax.set_xscale('log')
        ax.set_xlim(14, 700)
        ax.xaxis.set_major_locator(FixedLocator([20, 50, 100, 200, 500]))
        ax.xaxis.set_major_formatter(FixedFormatter(['20', '50', '100', '200', '500']))
        ax.xaxis.set_minor_locator(LogLocator(base=10, subs=[3, 4, 6, 7, 8, 9]))
        ax.set_ylim(ymin, ymax)
        ax.yaxis.set_major_locator(MultipleLocator(ystep))
        ax.yaxis.set_minor_locator(AutoMinorLocator(5))
        ax.tick_params(axis="both", which="major", direction="in", length=3.6)
        ax.tick_params(axis="both", which="minor", direction="in", length=2.2)
        ax.set_xlabel("T, K")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{tag} P = {P_bar} bar", loc="left", fontweight="bold")
        ax.grid(alpha=0.3)
        if tag == "(a)":
            ax.legend(fontsize=8)
    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(f"{OUTDIR}/{fname}", dpi=150)
    plt.close(fig)
    print(f"Saved {fname}")


make_4panel_with_overlay(
    get_density, "rho", r"$\rho$, kg/m$^3$",
    "pH2 density vs T -- this work vs TREND (Leachman EOS)",
    "fig6_density_vs_trend.png",
    y_specs=[(0, 0.20, 0.05), (0, 80, 20), (0, 80, 20), (0, 80, 20)],
)

make_4panel_with_overlay(
    get_Cp_over_R, "Cp", r"$C_{\mathrm{P}}/R$",
    "pH2 isobaric heat capacity vs T -- this work vs TREND (Leachman EOS)",
    "fig7_Cp_vs_trend.png",
    y_specs=[(2.2, 4.2, 0.5), (2.2, 4.2, 0.5), (0, 13, 2), (0, 14, 2)],
    divide_by_R=True,
)

make_4panel_with_overlay(
    get_Cv_over_R, "Cv", r"$C_{\mathrm{V}}/R$",
    "pH2 isochoric heat capacity vs T -- this work vs TREND (Leachman EOS)",
    "fig8_Cv_vs_trend.png",
    y_specs=[(1.2, 3.2, 0.5)] * 4,
    divide_by_R=True,
)

make_4panel_with_overlay(
    get_sonic, "ws", "sonic velocity, m/s",
    "pH2 sonic velocity vs T -- this work vs TREND (Leachman EOS)",
    "fig9_sonic_vs_trend.png",
    y_specs=[(100, 1900, 500), (100, 2100, 500), (100, 2200, 500), (100, 2300, 500)],
)

make_4panel_with_overlay(
    get_JT, "JT", r"$\mu_{\mathrm{JT}}$, K/bar",
    "pH2 Joule-Thomson coefficient vs T -- this work vs TREND (Leachman EOS)",
    "fig10_JT_vs_trend.png",
     y_specs=[(-0.2, 2.6, 0.5), (-0.3, 1.9, 0.5), (-0.2, 1.1, 0.2), (-0.2, 0.52, 0.1)],
)

print("\nAll comparison figures done.")

Saved fig6_density_vs_trend.png
Saved fig7_Cp_vs_trend.png
Saved fig8_Cv_vs_trend.png
Saved fig9_sonic_vs_trend.png
Saved fig10_JT_vs_trend.png

All comparison figures done.
